## 实验要求1：
利用深度学习方法,对各学科做一个排名模型,能够较好的预测出排名位置,
并且利用MSE、MAPE等指标来进行评价模型的优劣。

In [3]:
"""
实验要求1：利用深度学习方法,对各学科做一个排名模型,能够较好的预测出排名位置,
并且利用MSE、MAPE等指标来进行评价模型的优劣。
"""

import os
import math
import mysql.connector
from mysql.connector import Error
import numpy as np
import pandas as pd
from scipy.stats import spearmanr, rankdata
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt
from matplotlib import rcParams
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import warnings
import traceback

# ============================================================================
# ⚙️ 配置与初始化
# ============================================================================
warnings.filterwarnings('ignore')
rcParams['font.sans-serif'] = ['SimHei']
rcParams['axes.unicode_minus'] = False

# ============================================================================
# 🎛️ 超参数配置区域
# ============================================================================
class HyperParameters:
    """集中管理所有超参数"""
    DB_CONFIG = {
        'host': os.getenv('MYSQL_HOST', 'localhost'),
        'user': os.getenv('MYSQL_USER', 'root'),
        'password': os.getenv('MYSQL_PASSWORD', 'zzy419220'),
        'database': os.getenv('MYSQL_DB', 'university_ranking')
    }
    FEATURES = ['web_of_science_documents', 'cites', 'cites_per_paper', 'top_papers']
    TARGET = 'ranking_position'
    MIN_SAMPLES_PER_FIELD = 15
    OUTLIER_QUANTILE_LOW = 0.05
    OUTLIER_QUANTILE_HIGH = 0.95
    TEST_SIZE = 0.2
    VAL_SIZE = 0.2
    RANDOM_STATE = 42
    HIDDEN_LAYERS = [128, 64, 32]
    DROPOUT_RATE = 0.3
    ACTIVATION = 'leaky_relu'
    USE_BATCH_NORM = True
    BATCH_SIZE = 32
    EPOCHS = 200
    LEARNING_RATE = 0.001
    WEIGHT_DECAY = 1e-4
    OPTIMIZER = 'adamw'
    LR_SCHEDULER = 'plateau'
    LR_PATIENCE = 10
    LR_FACTOR = 0.5
    EARLY_STOP_PATIENCE = 20
    EARLY_STOP_MIN_DELTA = 0.001
    NORMALIZE_METRICS = True
    METRIC_NORM_METHOD = 'minmax'
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    USE_AMP = torch.cuda.is_available()
    NUM_WORKERS = 0
    PIN_MEMORY = torch.cuda.is_available()

HP = HyperParameters()

# ============================================================================
# 🧠 模型与辅助类定义
# ============================================================================
class RankingNet(nn.Module):
    """深度神经网络模型"""
    def __init__(self, input_dim):
        super(RankingNet, self).__init__()
        layers = []
        prev_dim = input_dim
        for hidden_dim in HP.HIDDEN_LAYERS:
            layers.append(nn.Linear(prev_dim, hidden_dim))
            if HP.USE_BATCH_NORM:
                layers.append(nn.BatchNorm1d(hidden_dim))
            if HP.ACTIVATION == 'relu':
                layers.append(nn.ReLU())
            elif HP.ACTIVATION == 'leaky_relu':
                layers.append(nn.LeakyReLU(0.1))
            if HP.DROPOUT_RATE > 0:
                layers.append(nn.Dropout(HP.DROPOUT_RATE))
            prev_dim = hidden_dim
        layers.append(nn.Linear(prev_dim, 1))
        self.network = nn.Sequential(*layers)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

    def forward(self, x):
        return self.network(x).squeeze(-1)

class EarlyStopping:
    """早停机制"""
    def __init__(self):
        self.patience = HP.EARLY_STOP_PATIENCE
        self.min_delta = HP.EARLY_STOP_MIN_DELTA
        self.counter = 0
        self.best_loss = None
        self.early_stop = False

    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.counter = 0

class MetricsNormalizer:
    """评估指标正则化器"""
    def __init__(self):
        self.scalers = {}
        self.fitted = False

    def fit(self, metrics_dict):
        if not metrics_dict: return
        all_metrics = {}
        for field_metrics in metrics_dict.values():
            for name, value in field_metrics.items():
                if name not in all_metrics: all_metrics[name] = []
                all_metrics[name].append(value)
        for name, values in all_metrics.items():
            scaler = MinMaxScaler() if HP.METRIC_NORM_METHOD == 'minmax' else StandardScaler()
            scaler.fit(np.array(values).reshape(-1, 1))
            self.scalers[name] = scaler
        self.fitted = True

    def transform(self, metrics):
        if not self.fitted or not HP.NORMALIZE_METRICS: return metrics
        normalized = {}
        for name, value in metrics.items():
            if name in self.scalers:
                normalized[name] = self.scalers[name].transform([[value]])[0][0]
            else:
                normalized[name] = value
        return normalized

    def get_composite_score(self, metrics):
        if not self.fitted or not HP.NORMALIZE_METRICS: return metrics.get('R2', 0)
        weights = {'MAE': -0.15, 'MSE': -0.15, 'RMSE': -0.15, 'R2': 0.30, 'MAPE': -0.15, 'Spearman': 0.10}
        score = 0
        for name, weight in weights.items():
            if name in metrics:
                norm_val = metrics[name]
                score += abs(weight) * (1 - norm_val) if weight < 0 else weight * norm_val
        return score

# ============================================================================
# 🚀 主控制器
# ============================================================================
class DeepLearningRankingPredictor:
    def __init__(self):
        self.results = {}
        self.metrics_normalizer = MetricsNormalizer()

    def plot_training_history(self, field_name, train_losses, val_losses, output_dir='results/dl_models_reloaded/plots'):
        """绘制训练过程的损失曲线"""
        os.makedirs(output_dir, exist_ok=True)

        plt.figure(figsize=(12, 6))
        epochs = range(1, len(train_losses) + 1)

        plt.plot(epochs, train_losses, 'b-', label='训练损失', linewidth=2)
        plt.plot(epochs, val_losses, 'orange', label='验证损失', linewidth=2)

        plt.title(f'{field_name} - 训练过程', fontsize=16, fontweight='bold')
        plt.xlabel('Epoch', fontsize=12)
        plt.ylabel('Loss (MSE)', fontsize=12)
        plt.legend(fontsize=12)
        plt.grid(True, alpha=0.3)
        plt.tight_layout()

        # 保存图片
        safe_filename = field_name.replace('/', '_').replace('\\', '_').replace(' ', '_')
        save_path = os.path.join(output_dir, f'{safe_filename}_training_history.png')
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.close()

        return save_path

    def load_data(self):
        print("=" * 80 + "\n📥 正在连接数据库并加载数据...")
        try:
            conn = mysql.connector.connect(**HP.DB_CONFIG)
            query = "SELECT * FROM university_rankings WHERE ranking_position IS NOT NULL"
            df = pd.read_sql(query, conn)
            conn.close()
            print(f"✅ 数据加载成功: {len(df):,} 条记录, {df['subject_field'].nunique()} 个学科")
            return df
        except mysql.connector.Error as err:
            print(f"❌ 数据库连接错误: {err}")
            return None

    def preprocess(self, df):
        print("=" * 80 + "\n🔧 数据预处理中...")
        for col in HP.FEATURES + [HP.TARGET]:
            df[col] = pd.to_numeric(df[col], errors='coerce')
        for col in HP.FEATURES:
            q_low, q_high = df[col].quantile([HP.OUTLIER_QUANTILE_LOW, HP.OUTLIER_QUANTILE_HIGH])
            df[col] = np.clip(df[col], q_low, q_high)
        original_fields = df['subject_field'].nunique()
        field_counts = df['subject_field'].value_counts()
        valid_fields = field_counts[field_counts >= HP.MIN_SAMPLES_PER_FIELD].index
        df = df[df['subject_field'].isin(valid_fields)]
        print(f"✅ 预处理完成: 保留 {len(valid_fields)}/{original_fields} 个学科")
        return df

    def calculate_metrics(self, y_true, y_pred):
        y_true, y_pred = np.array(y_true), np.array(y_pred)
        return {
            'MAE': mean_absolute_error(y_true, y_pred),
            'MSE': mean_squared_error(y_true, y_pred),
            'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
            'R2': r2_score(y_true, y_pred),
            'MAPE': np.mean(np.abs((y_true - y_pred) / (y_true + 1e-10))) * 100,
            'Spearman': spearmanr(rankdata(y_true), rankdata(y_pred))[0]
        }

    def train_single_model(self, X_train, y_train, X_val, y_val):
        train_dataset = TensorDataset(torch.FloatTensor(X_train), torch.FloatTensor(y_train.values))
        train_loader = DataLoader(train_dataset, batch_size=HP.BATCH_SIZE, shuffle=True, num_workers=HP.NUM_WORKERS, pin_memory=HP.PIN_MEMORY, drop_last=True)
        X_val_tensor, y_val_tensor = torch.FloatTensor(X_val).to(HP.DEVICE), torch.FloatTensor(y_val.values).to(HP.DEVICE)

        model = RankingNet(X_train.shape[1]).to(HP.DEVICE)
        criterion = nn.MSELoss()
        optimizer = optim.AdamW(model.parameters(), lr=HP.LEARNING_RATE, weight_decay=HP.WEIGHT_DECAY)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', factor=HP.LR_FACTOR, patience=HP.LR_PATIENCE)
        scaler = torch.cuda.amp.GradScaler(enabled=HP.USE_AMP)
        early_stopping = EarlyStopping()

        best_val_loss = float('inf')
        best_model_state = None

        # 记录训练和验证损失
        train_losses = []
        val_losses = []

        for epoch in range(HP.EPOCHS):
            model.train()
            epoch_train_loss = 0
            batch_count = 0
            for batch_X, batch_y in train_loader:
                batch_X, batch_y = batch_X.to(HP.DEVICE), batch_y.to(HP.DEVICE)
                optimizer.zero_grad()
                with torch.cuda.amp.autocast(enabled=HP.USE_AMP):
                    outputs = model(batch_X)
                    loss = criterion(outputs, batch_y)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                epoch_train_loss += loss.item()
                batch_count += 1

            # 计算平均训练损失
            avg_train_loss = epoch_train_loss / batch_count if batch_count > 0 else 0
            train_losses.append(avg_train_loss)

            model.eval()
            with torch.no_grad():
                with torch.cuda.amp.autocast(enabled=HP.USE_AMP):
                    val_loss = criterion(model(X_val_tensor), y_val_tensor).item()

            val_losses.append(val_loss)

            scheduler.step(val_loss)
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_model_state = model.state_dict().copy()

            early_stopping(val_loss)
            if early_stopping.early_stop:
                print(f"  提前停止于第 {epoch+1} 轮, 最佳验证损失: {best_val_loss:.4f}")
                break

        model.load_state_dict(best_model_state)
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        return model, train_losses, val_losses

    def train_all_models(self, df):
        print("=" * 80 + "\n🚀 开始训练所有模型...")
        fields = df['subject_field'].unique()
        all_raw_metrics = {}

        for i, field in enumerate(fields, 1):
            print(f"\n{'─' * 80}\n🔬 [{i:2d}/{len(fields)}] 训练学科: {field}")
            df_field = df[df['subject_field'] == field]
            X, y = df_field[HP.FEATURES], df_field[HP.TARGET]

            imputer = SimpleImputer(strategy='median')
            X_imputed = imputer.fit_transform(X)
            scaler = StandardScaler()
            X_scaled = scaler.fit_transform(X_imputed)

            X_temp, X_test, y_temp, y_test = train_test_split(X_scaled, y, test_size=HP.TEST_SIZE, random_state=HP.RANDOM_STATE)
            val_size_adj = HP.VAL_SIZE / (1 - HP.TEST_SIZE)
            X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=val_size_adj, random_state=HP.RANDOM_STATE)

            print(f"  📊 数据划分: 训练={len(X_train)} | 验证={len(X_val)} | 测试={len(X_test)}")

            try:
                model, train_losses, val_losses = self.train_single_model(X_train, y_train, X_val, y_val)
                model.eval()
                with torch.no_grad():
                    y_pred = model(torch.FloatTensor(X_test).to(HP.DEVICE)).cpu().numpy()

                metrics = self.calculate_metrics(y_test, y_pred)
                all_raw_metrics[field] = metrics

                # 绘制训练过程图
                plot_path = self.plot_training_history(field, train_losses, val_losses)

                self.results[field] = {'model': model, 'scaler': scaler, 'imputer': imputer, 'raw_metrics': metrics,
                                       'sizes': (len(X_train), len(X_val), len(X_test)),
                                       'plot_path': plot_path,
                                       'train_losses': train_losses,
                                       'val_losses': val_losses}

                status = "🟢" if metrics['R2'] > 0.5 else "🟡" if metrics['R2'] > 0 else "🔴"
                print(f"  {status} 训练完成: R²={metrics['R2']:.4f}, MAE={metrics['MAE']:.2f}, MAPE={metrics['MAPE']:.2f}%")
                print(f"  📊 训练过程图已保存: {plot_path}")
            except Exception:
                print(f"  ❌ 训练失败: {traceback.format_exc()}")

        if HP.NORMALIZE_METRICS and all_raw_metrics:
            print("=" * 80 + "\n📐 正在对评估指标进行正则化...")
            self.metrics_normalizer.fit(all_raw_metrics)
            for field, data in self.results.items():
                norm_metrics = self.metrics_normalizer.transform(data['raw_metrics'])
                data['normalized_metrics'] = norm_metrics
                data['composite_score'] = self.metrics_normalizer.get_composite_score(norm_metrics)
            print("✅ 指标正则化完成")

    def report_and_save(self, output_dir='results/dl_models_reloaded'):
        if not self.results:
            print("❌ 没有可用的训练结果")
            return
        
        os.makedirs(output_dir, exist_ok=True)
        print("=" * 80 + "\n📈 模型性能摘要报告" + "=" * 80)
        
        sort_key = 'composite_score' if HP.NORMALIZE_METRICS else 'R2'
        reverse_sort = True if sort_key == 'composite_score' or sort_key == 'R2' else False
        
        # 修正：确保在排序前 'composite_score' 存在
        for field, data in self.results.items():
            if 'composite_score' not in data:
                data['composite_score'] = data['raw_metrics'].get('R2', 0)

        sorted_results = sorted(self.results.items(), key=lambda x: x[1][sort_key], reverse=reverse_sort)

        print(f"\n🏆 表现最佳的前5个学科 (按 {sort_key} 排序):")
        for field, data in sorted_results[:5]:
            m = data['raw_metrics']
            score_info = f"| 综合评分: {data['composite_score']:.4f}" if HP.NORMALIZE_METRICS else ""
            print(f"  - {field:<35} | R²: {m['R2']:.4f} | MAE: {m['MAE']:.2f} {score_info}")

        print(f"\n⚠️ 需要改进的后5个学科 (按 {sort_key} 排序):")
        for field, data in sorted_results[-5:]:
            m = data['raw_metrics']
            score_info = f"| 综合评分: {data['composite_score']:.4f}" if HP.NORMALIZE_METRICS else ""
            print(f"  - {field:<35} | R²: {m['R2']:.4f} | MAE: {m['MAE']:.2f} {score_info}")

        results_data = []
        for field, data in self.results.items():
            row = {'subject_field': field, 'train_size': data['sizes'][0], 'val_size': data['sizes'][1], 'test_size': data['sizes'][2]}
            row.update({k: round(v, 4) for k, v in data['raw_metrics'].items()})
            if HP.NORMALIZE_METRICS:
                row['composite_score'] = round(data['composite_score'], 4)
            results_data.append(row)
        
        df_results = pd.DataFrame(results_data).sort_values(sort_key, ascending=not reverse_sort)
        csv_path = os.path.join(output_dir, 'dl_model_metrics.csv')
        df_results.to_csv(csv_path, index=False, encoding='utf-8-sig')
        print(f"\n💾 结果已保存到: {csv_path}")

# ============================================================================
# 🚀 主执行函数
# ============================================================================
def main():
    print("🎯" * 20 + " 深度学习排名预测系统 v2.1 " + "🎯" * 20)
    print(f"🖥️ 使用设备: {HP.DEVICE}{' (⚡已启用混合精度)' if HP.USE_AMP else ''}")
    
    predictor = DeepLearningRankingPredictor()
    df = predictor.load_data()
    if df is None: return
    
    df_processed = predictor.preprocess(df)
    if df_processed is None or df_processed.empty:
        print("❌ 预处理后无有效数据，程序终止。")
        return
        
    predictor.train_all_models(df_processed)
    predictor.report_and_save()
    
    print("\n" + "✅" * 20 + " 所有任务执行完毕 " + "✅" * 20)

if __name__ == "__main__":
    main()


🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯 深度学习排名预测系统 v2.1 🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯🎯
🖥️ 使用设备: cuda (⚡已启用混合精度)
📥 正在连接数据库并加载数据...
✅ 数据加载成功: 34,121 条记录, 22 个学科
🔧 数据预处理中...
✅ 预处理完成: 保留 22/22 个学科
🚀 开始训练所有模型...

────────────────────────────────────────────────────────────────────────────────
🔬 [ 1/22] 训练学科: AGRICULTURAL SCIENCES
  📊 数据划分: 训练=828 | 验证=276 | 测试=277
  🟢 训练完成: R²=0.7385, MAE=148.94, MAPE=29.27%
  📊 训练过程图已保存: results/dl_models_reloaded/plots\AGRICULTURAL_SCIENCES_training_history.png

────────────────────────────────────────────────────────────────────────────────
🔬 [ 2/22] 训练学科: BIOLOGY & BIOCHEMISTRY
  📊 数据划分: 训练=989 | 验证=330 | 测试=330
  🟢 训练完成: R²=0.9221, MAE=111.59, MAPE=27.81%
  📊 训练过程图已保存: results/dl_models_reloaded/plots\BIOLOGY_&_BIOCHEMISTRY_training_history.png

────────────────────────────────────────────────────────────────────────────────
🔬 [ 3/22] 训练学科: CHEMISTRY
  📊 数据划分: 训练=1284 | 验证=428 | 测试=429
  🟢 训练完成: R²=0.9713, MAE=89.03, MAPE=14.00%
  📊 训练过程图已保存: results/dl_models_reloaded/plots\CHEMI